# 09 Counterfactual View Expert Probe

05 failed because pseudoGT routing became an easy-domain filter.  This
notebook tests a different pseudoGT x MoE idea:

**Do not create experts by preserving a domain bucket.  Create experts
by the view condition that made a pseudo box appear.**

A night box that only appears after illumination enhancement is not
merely a low-confidence night sample.  It is an
`illumination_rescued` pseudoGT expert sample.

## What This Runs

The bounded probe uses the 03 BN-residual DQA aggregate checkpoint.

For each sampled client image it predicts six views:

- original
- original horizontal flip
- brightness enhanced
- brightness enhanced horizontal flip
- CLAHE enhanced
- CLAHE enhanced horizontal flip

Boxes are clustered back in original coordinates and split into:

- `clean_original`: stable in original views
- `illumination_rescued`: not stable in original, stable in enhanced views
- `cross_view_bridge`: appears in both original and enhanced views, but not enough original views alone

The notebook optionally trains a short neck/head probe on the
`illumination_rescued` expert dataset.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "moe":
    MOE_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    MOE_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa" / "moe"
else:
    MOE_ROOT = cwd

SCENE_ROOT = MOE_ROOT.parent
WORKSPACE = MOE_ROOT / "output" / "09_counterfactual_view_expert_probe"
RUNNER = MOE_ROOT / "scripts" / "run_moe_09_counterfactual_view_expert_probe.py"

print("MOE_ROOT", MOE_ROOT)
print("WORKSPACE", WORKSPACE)
print("RUNNER", RUNNER)

## Execute Bounded Probe

In [ ]:
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--max-images-per-client", "80",
    "--train-probe",
    "--evaluate",
    "--notify",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=MOE_ROOT, check=True)

## Counterfactual View Statistics

In [ ]:
client_stats = pd.read_csv(WORKSPACE / "stats" / "09_view_expert_probe_client_stats.csv")
display(client_stats)

summary = json.loads((WORKSPACE / "stats" / "09_view_expert_probe_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary["totals"], indent=2, ensure_ascii=False))
print(json.dumps(summary["day_night_signal"], indent=2, ensure_ascii=False))

## Training Probe Metrics

In [ ]:
metrics_path = WORKSPACE / "stats" / "09_training_probe_metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    train_summary_path = WORKSPACE / "stats" / "09_training_probe_summary.json"
    print(train_summary_path.read_text(encoding="utf-8") if train_summary_path.exists() else "No training summary yet.")

## Report

In [ ]:
report = WORKSPACE / "09_counterfactual_view_expert_probe_report.md"
print(report)
print(report.read_text(encoding="utf-8")[:9000])